# Issue #254 — final source-schema integration

This notebook binds the accepted #250–#253 artifacts, rebuilds a representative multi-year historical-proxy plus strict-forward M×3 package, verifies deterministic replay and restore portability, and presents the downstream readiness decisions. It does not claim full-history model readiness.

In [1]:
import json
import os
import sys
from pathlib import Path
import pandas as pd

repo = Path.cwd()
sys.path.insert(0, str(repo / 'src'))
from physical_schema import resolve_data_root
from schema_integration import build_integration_package, verify_integration_package

data_root = resolve_data_root(os.environ.get('BEIJING_LOT_DATA_ROOT'))
package = build_integration_package(data_root)
receipt = verify_integration_package(package)
print('External root configured:', data_root.name)
print('Integration status:', receipt['status'])

External root configured: data
Integration status: PASS


In [2]:
summary = {
    'accepted_predecessors': receipt['predecessors'],
    'downstream_contracts': receipt['downstream_contracts'],
    'dictionary_fields': receipt['dictionary_fields'],
    **receipt['physical_receipt'],
    **receipt['replay_receipt'],
}
pd.DataFrame({'value': pd.Series(summary)})

,value
accepted_predecessors,4
downstream_contracts,5
dictionary_fields,79
status,PASS
issue_slots,8
issues,6
observations,16
feature_cells,60
available_feature_cells,58
missing_feature_cells,2


In [3]:
readiness = pd.read_csv(package / 'downstream_readiness.csv')
readiness[['issue', 'consumer', 'execution_readiness', 'model_data_readiness', 'deterministic_blocker_reason']]

,issue,consumer,execution_readiness,model_data_readiness,deterministic_blocker_reason
0,235,BJ_SP_RECOVERY,PI_USER_CHECKPOINT,RECOVERY_REQUIRED,Full historical native-line/SP panel is absent...
1,236,EU_ODDS_RECOVERY,PI_USER_CHECKPOINT,RECOVERY_REQUIRED,Bookmaker-level history and non-zero BJ-line t...
2,237,POWER_FEATURE_SUPPORT,PI_SCOPE_REVIEW,IDENTITY_GAP,Canonical real-world team/competition IDs rema...
3,248,FORWARD_SNAPSHOTS,READY_TO_IMPLEMENT,COLLECTION_REQUIRED,NaN
4,249,PORTABILITY,READY_TO_IMPLEMENT,PORTABILITY_REQUIRED,NaN


In [4]:
dictionary = pd.read_csv(package / 'data_dictionary.csv')
dictionary.groupby(['store', 'semantic_boundary'], dropna=False).size().rename('fields').reset_index()

,store,semantic_boundary,fields
0,control_sqlite,pre_event_control_or_feature,44
1,feature_parquet,pre_event_model_input,19
2,label_parquet,post_outcome_label,8
3,label_sqlite,post_outcome_label,8


In [5]:
assert receipt['replay_receipt']['byte_deterministic_replay']
assert receipt['replay_receipt']['restore_copy_verified']
assert len(receipt['replay_receipt']['historical_proxy_years']) >= 2
assert readiness.loc[readiness.issue.eq(235), 'model_data_readiness'].item() == 'RECOVERY_REQUIRED'
assert readiness.loc[readiness.issue.eq(237), 'execution_readiness'].item() == 'PI_SCOPE_REVIEW'
pd.DataFrame([
    {'question': 'Historical exploration', 'decision': 'Usable only in historical_proxy lane; not strict wall-clock evidence.'},
    {'question': 'Forward validation', 'decision': 'Use only strict_forward pre-kickoff poll rows under inherited schedule caveat.'},
    {'question': 'Next data work', 'decision': '#235/#236 recovery checkpoint; #248 collection and #249 portability can consume the interface.'},
    {'question': 'Identity limitation', 'decision': '#237 requires an explicit real-world team/competition identity recovery decision.'},
] )

,question,decision
0,Historical exploration,Usable only in historical_proxy lane; not stri...
1,Forward validation,Use only strict_forward pre-kickoff poll rows ...
2,Next data work,#235/#236 recovery checkpoint; #248 collection...
3,Identity limitation,#237 requires an explicit real-world team/comp...


## PI interpretation boundary

The integration succeeds as a data-interface checkpoint. It does not turn proxy horizons into strict timestamps, does not invent canonical real-world team identities, and does not make the unified model trainable on complete history. Closing #254 releases the recovery work; it does not declare that recovery complete.